In [ ]:
from lightglue import LightGlue, SuperPoint, DISK, SIFT
from lightglue.utils import load_image, rbd
from lightglue import viz2d
import torch

torch.set_grad_enabled(False)

# Test the image feature correspondence between images
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 'mps', 'cpu'

extractor = SuperPoint(max_num_keypoints=2048).eval().to(device)  # load the extractor
matcher = LightGlue(features="superpoint").eval().to(device)

In [ ]:
image0 = load_image("rendered_img_1.png")
image1 = load_image("gt_img_1.png")

feats0 = extractor.extract(image0.to(device))
feats1 = extractor.extract(image1.to(device))
matches01 = matcher({"image0": feats0, "image1": feats1})
feats0, feats1, matches01 = [
    rbd(x) for x in [feats0, feats1, matches01]
]  # remove batch dimension

kpts0, kpts1, matches = feats0["keypoints"], feats1["keypoints"], matches01["matches"]
m_kpts0, m_kpts1 = kpts0[matches[..., 0]], kpts1[matches[..., 1]]

axes = viz2d.plot_images([image0, image1])
viz2d.plot_matches(m_kpts0, m_kpts1, color="lime", lw=0.2)
viz2d.add_text(0, f'Stop after {matches01["stop"]} layers', fs=20)

kpc0, kpc1 = viz2d.cm_prune(matches01["prune0"]), viz2d.cm_prune(matches01["prune1"])
viz2d.plot_images([image0, image1])
viz2d.plot_keypoints([kpts0, kpts1], colors=[kpc0, kpc1], ps=10)

print(image0.shape)
print(m_kpts0.shape)
m_kpts0 = m_kpts0.cpu().numpy().astype(int)
m_kpts1 = m_kpts1.cpu().numpy().astype(int)
image0_cp = image0.clone()
image1_cp = image1.clone()
image0_cp[:, :, :] = 1
image0_cp[:, m_kpts0[:,1], m_kpts0[:,0]] = image0[:, m_kpts0[:,1], m_kpts0[:,0]]
image1_cp[:, :, :] = 1
image1_cp[:, m_kpts1[:,1], m_kpts1[:,0]] = image1[:, m_kpts1[:,1], m_kpts1[:,0]]
viz2d.plot_images([image0_cp, image1_cp])

In [ ]:
import numpy as np
import cv2
mask0 = torch.zeros(image0.shape[1], image0.shape[2]).to(int)
mask0[m_kpts0[:,1], m_kpts0[:,0]] = 1
mask0_np = mask0.cpu().numpy().astype(np.uint8)
kernel = np.ones((13, 13), np.uint8)
dilated_mask0 = cv2.dilate(mask0_np, kernel, iterations=1)

mask1 = torch.zeros(image1.shape[1], image1.shape[2]).to(int)
mask1[m_kpts1[:,1], m_kpts1[:,0]] = 1
mask1_np = mask1.cpu().numpy().astype(np.uint8)
dilated_mask1 = cv2.dilate(mask1_np, kernel, iterations=1)

dilated_mask0_2 = np.repeat(dilated_mask0[np.newaxis, :,: ], 3, axis=0)
dilated_mask1_2 = np.repeat(dilated_mask1[np.newaxis, :,: ], 3, axis=0)
image0_cp_2 = image0 * dilated_mask0_2
image1_cp_2 = image1 * dilated_mask1_2
viz2d.plot_images([image0_cp_2, image1_cp_2])

In [ ]:
import cv2 as cv
from matplotlib import pyplot as plt
# Add the ORB feature matching
img = cv.imread('rendered_img_1.png', cv.IMREAD_GRAYSCALE)
img_gt = cv.imread('gt_img_1.png', cv.IMREAD_GRAYSCALE)
orb = cv.ORB_create()
kp = orb.detect(img,None)
kp, des = orb.compute(img, kp)
img2 = cv.drawKeypoints(img, kp, None, color=(0,255,0), flags=0)

kp_gt = orb.detect(img_gt,None)
kp_gt, des_gt = orb.compute(img_gt, kp_gt)
img3 = cv.drawKeypoints(img_gt, kp_gt, None, color=(0,255,0), flags=0)

# show the two images
fig, ax = plt.subplots(1, 2, figsize=(20, 10))
ax[0].imshow(img2)
ax[1].imshow(img3)
plt.show()